# QUANT Interval Classifier — Sample Run on Standardized SWAN-SF Tensors

This notebook trains an **aeon `QUANTClassifier`** on a configurable sample of the standardized SWAN-SF tensor dataset.

It is designed as a safe first model-building notebook before running QUANT on the full dataset.

**Expected input files**

```text
X_train_standardized_aeon.npy   # shape: (n_train, 47, 60)
X_test_standardized_aeon.npy    # shape: (n_test, 47, 60)
y_train.npy                     # shape: (n_train,)
y_test.npy                      # shape: (n_test,)
```

**Important:** This notebook uses a sample for speed. Treat the results as a prototype, not the final scientific result.


## 1. Install and import packages

Run this cell in Colab or Jupyter. If `aeon` is already installed, the install command can be skipped.


In [ ]:
# Optional install for Google Colab / fresh environments
# You can comment this out if aeon is already installed.
!pip -q install aeon scikit-learn joblib pandas matplotlib

In [ ]:
from pathlib import Path
import gc
import inspect
import json
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from aeon.classification.interval_based import QUANTClassifier

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

## 2. Mount Google Drive and set paths

Update `BASE_DIR` so it points to the folder containing your standardized `.npy` files.


In [ ]:
# Google Colab only. If you are not using Colab, you can skip this cell.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Google Drive mount skipped:', e)

In [ ]:
# TODO: Change this path to your actual standardized dataset folder.
BASE_DIR = Path('/content/drive/MyDrive/solar_flare_forecasting/model_ready_partition_split/final_clean_magnetic_only')

X_TRAIN_PATH = BASE_DIR / 'X_train.npy'
X_TEST_PATH = BASE_DIR / 'X_test.npy'
Y_TRAIN_PATH = BASE_DIR / 'y_train.npy'
Y_TEST_PATH = BASE_DIR / 'y_test.npy'

OUTPUT_DIR = BASE_DIR / 'quant_sample_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for path in [X_TRAIN_PATH, X_TEST_PATH, Y_TRAIN_PATH, Y_TEST_PATH]:
    print(path, 'exists:', path.exists())

## 3. Configuration

Adjust these settings to control how much data is used.

Recommended first run:

- `TRAIN_SAMPLE_SIZE = 20000`
- `TEST_SAMPLE_SIZE = 20000`
- `INTERVAL_DEPTH = 1`
- `QUANTILE_DIVISOR = 8`

After the first run works, increase `TRAIN_SAMPLE_SIZE` before increasing model complexity.


In [ ]:
# Sampling settings
TRAIN_SAMPLE_SIZE = 20_000
TEST_SAMPLE_SIZE = 20_000       # Set to None to evaluate on the full test set.
USE_ALL_POSITIVES_IN_TRAIN_SAMPLE = True
POSITIVE_CLASS = 1

# QUANT settings. Keep these conservative for the first sample run.
INTERVAL_DEPTH = 1
QUANTILE_DIVISOR = 8

# Extra Trees settings used inside QUANT when the aeon version supports a custom estimator.
N_ESTIMATORS = 200
N_JOBS = -1                    # Use all available CPU cores for Extra Trees.

print('Train sample size:', TRAIN_SAMPLE_SIZE)
print('Test sample size:', TEST_SAMPLE_SIZE)
print('QUANT interval depth:', INTERVAL_DEPTH)
print('QUANT quantile divisor:', QUANTILE_DIVISOR)

## 4. Load the standardized tensors

The feature tensors are loaded with `mmap_mode='r'` so the entire arrays do not need to be copied into RAM immediately.

The sampled training and test subsets will be copied into RAM later.


In [ ]:
X_train = np.load(X_TRAIN_PATH, mmap_mode='r')
y_train = np.load(Y_TRAIN_PATH)

X_test = np.load(X_TEST_PATH, mmap_mode='r')
y_test = np.load(Y_TEST_PATH)

print('X_train:', X_train.shape, X_train.dtype)
print('y_train:', y_train.shape, y_train.dtype)
print('X_test :', X_test.shape, X_test.dtype)
print('y_test :', y_test.shape, y_test.dtype)

In [ ]:
def summarize_labels(y, name):
    y_int = y.astype(int)
    counts = pd.Series(y_int).value_counts().sort_index()
    summary = pd.DataFrame({
        'count': counts,
        'percent': 100 * counts / len(y_int),
    })
    print(f'\n{name} label distribution')
    display(summary)

summarize_labels(y_train, 'Training')
summarize_labels(y_test, 'Test')

# Basic shape checks for aeon format: (n_cases, n_channels, n_timepoints)
assert X_train.ndim == 3, 'X_train should be a 3D array: (n_cases, n_channels, n_timepoints).'
assert X_test.ndim == 3, 'X_test should be a 3D array: (n_cases, n_channels, n_timepoints).'
assert X_train.shape[0] == len(y_train), 'X_train and y_train have different numbers of cases.'
assert X_test.shape[0] == len(y_test), 'X_test and y_test have different numbers of cases.'
assert X_train.shape[1:] == X_test.shape[1:], 'Train/test feature shape mismatch.'

print('Feature shape per case:', X_train.shape[1:])

In [ ]:
# Quick data quality check on a small slice.
# This avoids scanning the entire memmapped array at this stage.
CHECK_N = min(1000, len(X_train), len(X_test))

train_slice = np.asarray(X_train[:CHECK_N])
test_slice = np.asarray(X_test[:CHECK_N])

print('Train slice has NaN:', np.isnan(train_slice).any())
print('Train slice has inf:', np.isinf(train_slice).any())
print('Test slice has NaN :', np.isnan(test_slice).any())
print('Test slice has inf :', np.isinf(test_slice).any())

## 5. Build train/test samples

Training sample strategy:

- Include all positive cases when possible.
- Fill the remaining sample slots with randomly selected negative cases.
- This gives the model enough flare examples while still keeping the run small.

Evaluation sample strategy:

- Random sample from the test set without rebalancing.
- This keeps the evaluation distribution closer to the real test distribution.


In [ ]:
def make_train_sample_indices(y, sample_size, positive_class=1, use_all_positives=True, random_state=42):
    rng = np.random.default_rng(random_state)
    y = np.asarray(y)

    pos_idx = np.flatnonzero(y == positive_class)
    neg_idx = np.flatnonzero(y != positive_class)

    if sample_size is None or sample_size >= len(y):
        idx = np.arange(len(y))
        rng.shuffle(idx)
        return idx

    if use_all_positives and len(pos_idx) < sample_size:
        n_pos = len(pos_idx)
        n_neg = sample_size - n_pos
        neg_sample = rng.choice(neg_idx, size=n_neg, replace=False)
        idx = np.concatenate([pos_idx, neg_sample])
    else:
        # Stratified fallback when positives do not fit or when use_all_positives=False.
        pos_rate = len(pos_idx) / len(y)
        n_pos = max(1, int(round(sample_size * pos_rate)))
        n_pos = min(n_pos, len(pos_idx))
        n_neg = sample_size - n_pos
        n_neg = min(n_neg, len(neg_idx))

        pos_sample = rng.choice(pos_idx, size=n_pos, replace=False)
        neg_sample = rng.choice(neg_idx, size=n_neg, replace=False)
        idx = np.concatenate([pos_sample, neg_sample])

    rng.shuffle(idx)
    return idx


def make_random_sample_indices(n, sample_size, random_state=42):
    rng = np.random.default_rng(random_state)
    if sample_size is None or sample_size >= n:
        return np.arange(n)
    return rng.choice(np.arange(n), size=sample_size, replace=False)

In [ ]:
train_idx = make_train_sample_indices(
    y_train,
    sample_size=TRAIN_SAMPLE_SIZE,
    positive_class=POSITIVE_CLASS,
    use_all_positives=USE_ALL_POSITIVES_IN_TRAIN_SAMPLE,
    random_state=RANDOM_STATE,
)

test_idx = make_random_sample_indices(
    len(y_test),
    sample_size=TEST_SAMPLE_SIZE,
    random_state=RANDOM_STATE + 1,
)

print('Train sample cases:', len(train_idx))
print('Test sample cases :', len(test_idx))

summarize_labels(y_train[train_idx], 'Training sample')
summarize_labels(y_test[test_idx], 'Test sample')

In [ ]:
def materialize_sample(X_mmap, y, idx, dtype=np.float32):
    # Sorting the indices makes reading from memmap more efficient.
    idx_sorted = np.sort(idx)
    X_sample = np.asarray(X_mmap[idx_sorted], dtype=dtype)
    y_sample = np.asarray(y[idx_sorted])

    # Shuffle after loading so the training order is random.
    shuffle_order = np.random.default_rng(RANDOM_STATE).permutation(len(idx_sorted))
    return X_sample[shuffle_order], y_sample[shuffle_order]

start = time.time()
X_fit, y_fit = materialize_sample(X_train, y_train, train_idx)
X_eval, y_eval = materialize_sample(X_test, y_test, test_idx)
load_time = time.time() - start

print(f'Sample materialization time: {load_time:.1f} seconds')
print('X_fit :', X_fit.shape, X_fit.dtype)
print('y_fit :', y_fit.shape, y_fit.dtype)
print('X_eval:', X_eval.shape, X_eval.dtype)
print('y_eval:', y_eval.shape, y_eval.dtype)

print('Approx X_fit RAM GB:', X_fit.nbytes / 1e9)
print('Approx X_eval RAM GB:', X_eval.nbytes / 1e9)

## 6. Train the QUANT classifier

This cell builds a QUANT model with conservative settings.

The notebook tries to use a custom `ExtraTreesClassifier` with `n_jobs=-1` when your installed aeon version supports the `estimator` argument. If not, it falls back to the standard `QUANTClassifier` constructor.


In [ ]:
def build_quant_classifier():
    quant_signature = inspect.signature(QUANTClassifier)
    quant_params = set(quant_signature.parameters.keys())

    kwargs = {
        'interval_depth': INTERVAL_DEPTH,
        'quantile_divisor': QUANTILE_DIVISOR,
    }

    if 'random_state' in quant_params:
        kwargs['random_state'] = RANDOM_STATE

    if 'estimator' in quant_params:
        estimator = ExtraTreesClassifier(
            n_estimators=N_ESTIMATORS,
            class_weight='balanced',
            n_jobs=N_JOBS,
            random_state=RANDOM_STATE,
        )
        kwargs['estimator'] = estimator
        print('Using custom ExtraTreesClassifier inside QUANT.')
    else:
        print('This aeon version does not expose estimator=. Using QUANT default estimator.')
        if 'class_weight' in quant_params:
            kwargs['class_weight'] = 'balanced'
            print('Using QUANT class_weight="balanced".')

    print('QUANT kwargs:', kwargs)
    return QUANTClassifier(**kwargs)

quant = build_quant_classifier()
quant

In [ ]:
start = time.time()
quant.fit(X_fit, y_fit)
fit_time = time.time() - start

print(f'QUANT fit time: {fit_time / 60:.2f} minutes')

## 7. Evaluate the sample model

For flare prediction, accuracy alone is not enough because the positive class is rare.

This notebook reports:

- confusion matrix
- TSS
- HSS
- balanced accuracy
- precision
- recall
- F1
- ROC-AUC
- PR-AUC

The threshold sweep is useful for diagnostics. For final reporting, tune the probability threshold on a validation partition and evaluate once on the held-out test partition.


In [ ]:
def safe_auc(metric_func, y_true, y_score):
    try:
        return metric_func(y_true, y_score)
    except ValueError:
        return np.nan


def binary_flare_metrics(y_true, y_score, threshold=0.5, positive_class=1):
    y_true = np.asarray(y_true).astype(int)
    y_pred = (np.asarray(y_score) >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    tpr = tp / (tp + fn) if (tp + fn) else 0.0
    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    tss = tpr - fpr

    hss_num = 2 * (tp * tn - fp * fn)
    hss_den = ((tp + fn) * (fn + tn)) + ((tp + fp) * (fp + tn))
    hss = hss_num / hss_den if hss_den else 0.0

    return {
        'threshold': threshold,
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
        'tss': tss,
        'hss': hss,
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'roc_auc': safe_auc(roc_auc_score, y_true, y_score),
        'pr_auc': safe_auc(average_precision_score, y_true, y_score),
    }

In [ ]:
start = time.time()
proba = quant.predict_proba(X_eval)
predict_time = time.time() - start
print(f'Prediction time: {predict_time / 60:.2f} minutes')

# Get probability column for the positive class.
classes = list(quant.classes_)
print('Model classes:', classes)
positive_col = classes.index(POSITIVE_CLASS)
flare_prob = proba[:, positive_col]

metrics_05 = binary_flare_metrics(y_eval, flare_prob, threshold=0.5)
pd.DataFrame([metrics_05]).T.rename(columns={0: 'value'})

In [ ]:
thresholds = np.linspace(0.01, 0.99, 99)
metrics_df = pd.DataFrame([
    binary_flare_metrics(y_eval, flare_prob, threshold=float(t))
    for t in thresholds
])

best_by_tss = metrics_df.sort_values('tss', ascending=False).head(10)
best_by_f1 = metrics_df.sort_values('f1', ascending=False).head(10)

print('Top thresholds by TSS')
display(best_by_tss)

print('Top thresholds by F1')
display(best_by_f1)

In [ ]:
best_threshold = float(best_by_tss.iloc[0]['threshold'])
best_metrics = binary_flare_metrics(y_eval, flare_prob, threshold=best_threshold)

print('Best diagnostic threshold by TSS:', best_threshold)
pd.DataFrame([best_metrics]).T.rename(columns={0: 'value'})

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(metrics_df['threshold'], metrics_df['tss'], label='TSS')
plt.plot(metrics_df['threshold'], metrics_df['precision'], label='Precision')
plt.plot(metrics_df['threshold'], metrics_df['recall'], label='Recall')
plt.plot(metrics_df['threshold'], metrics_df['f1'], label='F1')
plt.xlabel('Probability threshold')
plt.ylabel('Metric value')
plt.title('Threshold Diagnostic Curves')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 8. Save outputs

This saves:

- trained QUANT sample model
- metrics at threshold 0.5
- best diagnostic TSS-threshold metrics
- threshold sweep table
- prediction probabilities for the evaluation sample


In [ ]:
run_name = f'quant_sample_n{len(y_fit)}_depth{INTERVAL_DEPTH}_qdiv{QUANTILE_DIVISOR}'

model_path = OUTPUT_DIR / f'{run_name}.joblib'
metrics_path = OUTPUT_DIR / f'{run_name}_metrics.json'
thresholds_path = OUTPUT_DIR / f'{run_name}_threshold_sweep.csv'
preds_path = OUTPUT_DIR / f'{run_name}_eval_predictions.csv'

joblib.dump(quant, model_path)

metrics_payload = {
    'run_name': run_name,
    'train_sample_size': int(len(y_fit)),
    'eval_sample_size': int(len(y_eval)),
    'interval_depth': INTERVAL_DEPTH,
    'quantile_divisor': QUANTILE_DIVISOR,
    'n_estimators': N_ESTIMATORS,
    'fit_time_seconds': fit_time,
    'predict_time_seconds': predict_time,
    'metrics_threshold_0_5': metrics_05,
    'best_diagnostic_tss_threshold': best_threshold,
    'best_diagnostic_tss_metrics': best_metrics,
}

with open(metrics_path, 'w') as f:
    json.dump(metrics_payload, f, indent=2)

metrics_df.to_csv(thresholds_path, index=False)

preds_df = pd.DataFrame({
    'sample_position': np.arange(len(y_eval)),
    'y_true': y_eval.astype(int),
    'flare_probability': flare_prob,
})
preds_df.to_csv(preds_path, index=False)

print('Saved model to      :', model_path)
print('Saved metrics to    :', metrics_path)
print('Saved thresholds to :', thresholds_path)
print('Saved predictions to:', preds_path)